## 1. Setup and Data Loading
First, we import the necessary libraries and load the raw dataset.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

print("Libraries loaded successfully.")

In [ ]:
# Load the raw e-commerce dataset
# Note: Place your CSV file in the data/raw/ directory
DATA_PATH = '../data/raw/ecommerce_data.csv'

try:
    df = pd.read_csv(DATA_PATH, encoding='ISO-8859-1')
    print(f"Dataset loaded successfully: {len(df):,} records")
except FileNotFoundError:
    print(f"Error: Please place your dataset at {DATA_PATH}")
    print("Expected columns: InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country")

## 2. Dataset Overview
Understanding the structure and size of our data is the first step in any analysis. 
This helps identify what information is available and how it can support business questions.

In [ ]:
# Dataset dimensions
print("=" * 50)
print("DATASET DIMENSIONS")
print("=" * 50)
print(f"Total Records (Rows): {df.shape[0]:,}")
print(f"Total Features (Columns): {df.shape[1]}")
print("\n")

# Column names
print("Available Columns:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i}. {col}")

In [ ]:
# Data types and memory usage
print("=" * 50)
print("DATA TYPES & MEMORY USAGE")
print("=" * 50)
print(df.dtypes)
print(f"\nTotal Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
# Preview first few records
print("=" * 50)
print("SAMPLE RECORDS (First 5 Rows)")
print("=" * 50)
df.head()

## 3. Missing Values Analysis
Missing data can significantly impact analysis quality. From a business perspective:
- Missing **CustomerID** means we cannot attribute transactions to specific customers (affects customer-level analytics)
- Missing **Description** may indicate data entry issues or system errors

Understanding these gaps helps determine data quality and cleaning strategies.

In [ ]:
# Missing values analysis
print("=" * 50)
print("MISSING VALUES ANALYSIS")
print("=" * 50)

missing_data = pd.DataFrame({
    'Column': df.columns,
    'Missing Count': df.isnull().sum().values,
    'Missing Percentage': (df.isnull().sum().values / len(df) * 100).round(2)
})

missing_data = missing_data.sort_values('Missing Percentage', ascending=False)
missing_data = missing_data[missing_data['Missing Count'] > 0]

if len(missing_data) > 0:
    print(missing_data.to_string(index=False))
else:
    print("No missing values found in the dataset.")

In [ ]:
# Visualize missing values
if df.isnull().sum().sum() > 0:
    fig, ax = plt.subplots(figsize=(10, 4))
    missing_pct = (df.isnull().sum() / len(df) * 100)
    missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=True)
    
    missing_pct.plot(kind='barh', ax=ax, color='coral')
    ax.set_xlabel('Missing Percentage (%)')
    ax.set_title('Missing Values by Column')
    
    for i, v in enumerate(missing_pct.values):
        ax.text(v + 0.5, i, f'{v:.1f}%', va='center')
    
    plt.tight_layout()
    plt.show()
else:
    print("No missing values to visualize.")

## 4. Statistical Summary
Descriptive statistics provide insights into the distribution of numerical variables:
- **Quantity**: Understanding order sizes and detecting anomalies (negative values = returns)
- **UnitPrice**: Price distribution across products
- **CustomerID**: Customer coverage in the dataset

In [ ]:
# Statistical summary for numerical columns
print("=" * 50)
print("NUMERICAL COLUMNS - DESCRIPTIVE STATISTICS")
print("=" * 50)
df.describe()

In [ ]:
# Categorical columns summary
print("=" * 50)
print("CATEGORICAL COLUMNS - VALUE COUNTS")
print("=" * 50)

categorical_cols = df.select_dtypes(include=['object']).columns

for col in categorical_cols:
    unique_count = df[col].nunique()
    print(f"\n{col}:")
    print(f"  Unique Values: {unique_count:,}")
    if unique_count <= 10:
        print(f"  Values: {df[col].unique()[:10]}")
    else:
        print(f"  Top 5 Values:")
        print(df[col].value_counts().head().to_string())

## 5. Data Quality Checks
Before proceeding with analysis, we need to identify potential data quality issues:
- **Negative quantities** typically represent product returns
- **Zero or negative prices** may indicate promotional items or data errors
- **Duplicate transactions** could inflate metrics

In [ ]:
# Data quality checks
print("=" * 50)
print("DATA QUALITY CHECKS")
print("=" * 50)

# Check for negative quantities (returns)
if 'Quantity' in df.columns:
    negative_qty = (df['Quantity'] < 0).sum()
    print(f"Negative Quantity Records (Returns): {negative_qty:,} ({negative_qty/len(df)*100:.2f}%)")

# Check for zero or negative prices
if 'UnitPrice' in df.columns:
    zero_price = (df['UnitPrice'] <= 0).sum()
    print(f"Zero/Negative Price Records: {zero_price:,} ({zero_price/len(df)*100:.2f}%)")

# Check for duplicates
duplicate_count = df.duplicated().sum()
print(f"Duplicate Records: {duplicate_count:,} ({duplicate_count/len(df)*100:.2f}%)")

# Check CustomerID coverage
if 'CustomerID' in df.columns:
    customer_coverage = df['CustomerID'].notna().sum() / len(df) * 100
    print(f"Records with CustomerID: {customer_coverage:.2f}%")

## 6. Key Business Metrics Overview
Quick snapshot of high-level metrics to understand the business scale.

In [ ]:
# Calculate basic business metrics
print("=" * 50)
print("KEY BUSINESS METRICS (Raw Data)")
print("=" * 50)

# Filter valid transactions for metrics
valid_df = df[(df.get('Quantity', pd.Series([1]*len(df))) > 0) & 
              (df.get('UnitPrice', pd.Series([1]*len(df))) > 0)].copy()

if 'Quantity' in df.columns and 'UnitPrice' in df.columns:
    valid_df['Revenue'] = valid_df['Quantity'] * valid_df['UnitPrice']
    total_revenue = valid_df['Revenue'].sum()
    print(f"Total Revenue: ${total_revenue:,.2f}")

if 'InvoiceNo' in df.columns:
    total_orders = df['InvoiceNo'].nunique()
    print(f"Total Orders: {total_orders:,}")

if 'CustomerID' in df.columns:
    total_customers = df['CustomerID'].nunique()
    print(f"Unique Customers: {total_customers:,}")

if 'StockCode' in df.columns:
    total_products = df['StockCode'].nunique()
    print(f"Unique Products: {total_products:,}")

if 'Country' in df.columns:
    total_countries = df['Country'].nunique()
    print(f"Countries Served: {total_countries:,}")

## 7. Next Steps

Based on this exploration, the following steps are recommended:

1. **Data Cleaning** (see `src/data_cleaning.py`):
   - Handle missing CustomerID values
   - Convert InvoiceDate to datetime format
   - Remove or flag cancelled orders (negative quantities)
   - Filter out invalid price records

2. **Analysis** (see `src/analysis.py`):
   - Calculate revenue metrics
   - Analyze monthly sales trends
   - Identify top customers by monetary value

3. **Dashboard** (see `app/app.py`):
   - Build interactive Streamlit dashboard for stakeholders